# Door Segmentation: Claude Vision + SAM

Uses Claude (Bedrock) to identify the door bounding box, then SAM for pixel-perfect mask.

**Why:** Claude understands scene context — it can pick the right door even in cluttered side views where object detectors fail.

**Cost:** ~$0.06 for all 4 images.

In [ ]:
!pip install -q \
    "segment-anything @ git+https://github.com/facebookresearch/segment-anything.git" \
    opencv-python-headless pillow matplotlib boto3

In [ ]:
import os
os.makedirs("weights", exist_ok=True)
!wget -q -nc -P weights https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

In [ ]:
import torch
import numpy as np
import cv2
import json
import base64
import boto3
from PIL import Image
import matplotlib.pyplot as plt
from segment_anything import sam_model_registry, SamPredictor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Load SAM
sam = sam_model_registry["vit_b"](checkpoint="weights/sam_vit_b_01ec64.pth").to(DEVICE)
sam_predictor = SamPredictor(sam)

# Bedrock client
bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")
print("Models ready.")

In [ ]:
IMAGE_DIR = "doors"
os.makedirs(IMAGE_DIR, exist_ok=True)
images = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
print(f"Found {len(images)} images: {images}")

In [ ]:
def get_door_bbox_claude(image_path):
    """Ask Claude to return bounding box [x1, y1, x2, y2] of the main door."""
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode()
    
    # Get image dimensions
    img = Image.open(image_path)
    w, h = img.size

    response = bedrock.invoke_model(
        modelId="anthropic.claude-sonnet-4-20250514",
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 200,
            "messages": [{"role": "user", "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": img_b64}},
                {"type": "text", "text": f"This photo shows a door in a showroom. Identify the main door being photographed (the one most centered/prominent). Return its bounding box as pixel coordinates. Image is {w}x{h} pixels. Return ONLY JSON: {{\"box\": [x1, y1, x2, y2]}} where x1,y1 is top-left and x2,y2 is bottom-right. No explanation."}
            ]}]
        })
    )
    
    body = json.loads(response["body"].read())
    text = body["content"][0]["text"]
    # Parse JSON from response
    result = json.loads(text)
    box = result["box"]
    print(f"  Claude bbox: {box}")
    return np.array(box, dtype=np.float32)

In [ ]:
def segment_door(image_path):
    """Get door mask: Claude for bbox, SAM for precise segmentation."""
    image_np = np.array(Image.open(image_path).convert("RGB"))
    
    # Step 1: Claude identifies the door
    box = get_door_bbox_claude(image_path)
    
    # Step 2: SAM segments precisely within that box
    sam_predictor.set_image(image_np)
    mask, _, _ = sam_predictor.predict(box=box, multimask_output=False)
    
    return image_np, mask[0], box

In [ ]:
def show_result(image, mask, box):
    """Display original, bbox, and final cutout."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original
    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    # With bbox
    img_box = image.copy()
    x1, y1, x2, y2 = box.astype(int)
    cv2.rectangle(img_box, (x1, y1), (x2, y2), (0, 255, 0), 3)
    axes[1].imshow(img_box)
    axes[1].set_title("Claude bbox")
    axes[1].axis("off")
    
    # Cutout
    result = image.copy()
    result[~mask] = 255
    axes[2].imshow(result)
    axes[2].set_title("SAM mask")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Test on first image
if images:
    test_path = os.path.join(IMAGE_DIR, images[0])
    print(f"Processing: {images[0]}")
    image, mask, box = segment_door(test_path)
    show_result(image, mask, box)

In [ ]:
# Process all images and save cutouts
os.makedirs("output", exist_ok=True)

for img_name in images:
    img_path = os.path.join(IMAGE_DIR, img_name)
    print(f"\nProcessing: {img_name}")
    
    image, mask, box = segment_door(img_path)
    show_result(image, mask, box)
    
    # Save cutout with alpha
    rgba = np.dstack([image, (mask * 255).astype(np.uint8)])
    out_name = os.path.splitext(img_name)[0] + "_cutout.png"
    Image.fromarray(rgba).save(os.path.join("output", out_name))
    print(f"  Saved: output/{out_name}")

## Next Steps

Once cutouts look good:
1. Perspective-correct each face (homography)
2. Build box-shaped GLB with front/back/side textures
3. View in AR